## Initialization

In [ ]:
# Imports

from math import exp
from itertools import starmap
from pathlib import Path
from typing import Callable, TypeVar, Any, Literal
import pickle

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d
from scipy.signal import deconvolve

from data_processing.arc_paths import (
    get_parq_root, get_exp_root, INPUT_DATA_FOLDER
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.dataframe_validation import DetectorDataframeColumn
from data_processing.experiment_data_keys import ExperimentDataKey
from data_processing.helpers import stop, get_input_with_default
from data_processing.loading.dataframe_loading import load_psd
from data_processing.loading.timetag_processing import calculate_timetag_hours
from data_processing.processing.bimodal_fitting import (
    get_psd_energy_histogram,
    scan_histogram_slices,
    find_failed_slices,
    BimodalBounds,
    BimodalParams
)
from data_processing.processing.calibration import Detector, recalibrate
from data_processing.processing.figure_of_merit import gaussian
from data_processing.processing.neutron_classification import classify
from data_processing.processing.neutron_window_generation import (
    generate_nasa_neutron_window,
    generate_n_distro_neutron_window
)
from data_processing.types import NasaGenerationSettings, WindowType, NeutronWindowSettings
from data_processing.reporting.plotting import plot_classification
from data_processing.helpers import (
    stop,
    get_input_with_default,
    input_experiment_ids,
    get_midpoints_from_min_max_series
)
from data_processing.loading.window_loading import (
    load_side_borders, get_neutron_window_paths)
from data_processing.processing.neutron_window_strategy.strategy_factory import NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy import AbstractNeutronStrategy

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

## Functions

## Run Settings

In [ ]:
exp_id = "ID-423"
exp_data_filename = f"{exp_id}-n_spectrum.csv"
exp_data_path = Path() / exp_data_filename

sim_data_filenames = [
    "output_sigma_0.5MeV_wresolution.txt",
    # "output_14.1MeV.txt",
    # "output_14.1MeV_1e4.txt",
    # "output_14.1MeV_1e5.txt",
    # "output_4MeV_1e5.txt",
    # "output_8MeV_1e5.txt"
]
sim_data_description = "0.5MeV variability"
sim_data_paths = {filename: Path() / filename
                  for filename in sim_data_filenames}

## Loading

### Experimental Data

In [ ]:
exp_df = pd.read_csv(
    exp_data_path, 
    usecols=[
        "Light output bin start (MeVee)",
        "Light output bin end (MeVee)",
        "Neutron counts"
    ]
)

In [ ]:
midpoints = get_midpoints_from_min_max_series(
    exp_df["Light output bin start (MeVee)"],
    exp_df["Light output bin end (MeVee)"],
    exp_df.index
)
exp_df["Light output (MeVee)"] = midpoints

### Simulation Data

In [ ]:
sim_dfs = {}
for sim_filename, sim_data_path in sim_data_paths.items():
    sim_df = pd.read_fwf(sim_data_path)
    sim_df = sim_df[["NPS", "det_pulse (MeVee)"]].copy()
    sim_df.columns = ["Count rate", "Neutron light output (MeVee)"]
    sim_dfs[sim_filename] = sim_df

In [ ]:
sim_l_cut_dfs = {}
bins_lo = exp_df["Light output bin start (MeVee)"]
bins_hi = exp_df["Light output bin end (MeVee)"]
bins = pd.IntervalIndex.from_arrays(bins_lo, bins_hi)
for sim_filename, sim_df in sim_dfs.items():
    light_output_cut = pd.cut(sim_df["Neutron light output (MeVee)"],
                              bins=bins)
    sim_l_cut_dfs[sim_filename] = light_output_cut

In [ ]:
binned_sim_dfs = {}
for sim_filename, sim_l_cut_df in sim_l_cut_dfs.items():
    binned_sim_df = sim_df.groupby(sim_l_cut_df).sum()[["Count rate"]].copy()
    binned_sim_energy_bins = binned_sim_df.index.to_series()
    midpoints = binned_sim_energy_bins.apply(lambda x: x.mid)
    binned_sim_df["Light output (MeVee)"] = midpoints
    binned_sim_dfs[sim_filename] = binned_sim_df

## Normalization

In [ ]:
counts = exp_df['Neutron counts']
exp_max = counts.max()
print(exp_max)
exp_df['Counts (normalized)'] = counts / exp_max

In [ ]:
sim_max = 0
for binned_sim_df in binned_sim_dfs.values():
    counts = binned_sim_df['Count rate']
    this_sim_max = counts.max()
    sim_max = this_sim_max if this_sim_max > sim_max else sim_max
print(sim_max)
for binned_sim_df in binned_sim_dfs.values():
    counts = binned_sim_df['Count rate']
    binned_sim_df['Counts (normalized)'] = counts / sim_max

## Plotting

In [ ]:
sim_xys = {}
for sim_filename, binned_sim_df in binned_sim_dfs.items():
    sim_x = binned_sim_df["Light output (MeVee)"].astype(float) * 1000
    sim_y = binned_sim_df["Counts (normalized)"].astype(float)
    sim_xys[sim_filename] = (sim_x, sim_y)

exp_x = exp_df["Light output (MeVee)"] * 1000
exp_y = exp_df["Counts (normalized)"]

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(8, 5))
# fig.tight_layout()

for sim_filename, sim_xy in sim_xys.items():
    sim_x, sim_y = sim_xy
    axs.plot(
        sim_x,
        sim_y,
        ls='-',
        marker='',
        ms=5,
        # color='#424242FF',
        label="Simulation"
    )
axs.plot(
    exp_x,
    exp_y,
    ls='--',
    label="Experiment"
)

# axs.set_title('Beam Loading', fontsize = 16)
axs.set_xlim(-20, 1400)
# axs.set_ylim(-5,180)
axs.set_ylabel('Normalized counts', fontsize=14)
# axs.set_yscale("log")
axs.set_xlabel('Light output (keVee)', fontsize=14)
# axs.axhline(64, color = 'black', ls = "--", alpha = 0.7)
# axs.axhline(45, color = 'black', ls = "--", alpha = 0.7)
axs.tick_params(axis="x", labelsize=12)
# axs.annotate(
#     'Experiment',
#     (700, 0.15),
#     xytext=None,
#     xycoords='data',
#     textcoords='data',
#     fontsize=14
# )
# axs.annotate(
#     'Simulation',
#     (400, 0.05),
#     xytext=None,
#     xycoords='data',
#     textcoords='data',
#     fontsize=14,
#     color='C0'
# )
axs.legend(
    # loc=(0.78, 0.85)
)
fig.tight_layout()

base_file_name = f"PHD Sim {sim_data_description} vs Exp {exp_id}"
main_path = Path()
fig.savefig(main_path / f"{base_file_name}.png", format="png")
fig.savefig(main_path / f"{base_file_name}.pdf", format="pdf")

plt.show()

In [ ]:
fig, axs = plt.subplots(
    1, 1,
    figsize=(8, 5)
)

for sim_filename, sim_xy in sim_xys.items():
    sim_x, sim_y = sim_xy
    axs.plot(
        sim_x,
        sim_y,
        ls='-',
        marker='',
        ms=5,
        # color='#424242FF',
        label="Simulation"
    )
axs.plot(
    exp_x,
    exp_y,
    ls='--',
    label="Experiment"
)

# axs.set_title('Beam Loading', fontsize = 16)
axs.set_xlim(-20, 1400)
# axs.set_ylim(-5,180)
axs.set_ylabel('Normalized counts', fontsize=14)
axs.set_yscale("log")
axs.set_xlabel('Light output (keVee)', fontsize=14)
# axs.axhline(64, color = 'black', ls = "--", alpha = 0.7)
# axs.axhline(45, color = 'black', ls = "--", alpha = 0.7)
axs.tick_params(axis="x", labelsize=12)
# axs.annotate(
#     'Experiment',
#     (700,0.15),
#     xytext = None,
#     xycoords = 'data',
#     textcoords = 'data',
#     fontsize = 14
# )
# axs.annotate(
#     'Simulation',
#     (300, 0.0),
#     xytext=None,
#     xycoords='data',
#     textcoords='data',
#     fontsize=14,
#     color='C0'
# )
axs.legend(
    # loc=(0.8, 0.85)
)
fig.tight_layout()

base_file_name = f"PHD Sim {sim_data_description} vs Exp {exp_id} Log Scale"
main_path = Path()
fig.savefig(main_path / f"{base_file_name}.png", format="png")
fig.savefig(main_path / f"{base_file_name}.pdf", format="pdf")

plt.show()